1. Load Dataset
2. eng-> sentence ko  convert -> Embedding mein
 *  tokenization for unique word
3. Build RNN
4. Train
5. Prediction

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df=pd.read_csv('100_Unique_QA_Dataset.csv')

In [ ]:
df.shape

(90, 2)

In [ ]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [ ]:
# Tokenize
def tokenize(text):
    text= text.lower()
    text= text.replace('.','')
    text= text.replace(',','')
    text= text.replace('?','')
    text= text.replace('!','')
    return text.split()


In [ ]:
tokenize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [ ]:
# Vocab To gather unique word
vocab={'<UNK>':0}


In [ ]:
def build_vocab(row):
   tokenized_question= tokenize(row['question'])
   tokenized_answer= tokenize(row['answer'])

   merged_tokens= tokenized_question + tokenized_answer

   for token in merged_tokens:
      if token not in vocab:
         vocab[token]= len(vocab)
   print(tokenized_question,tokenized_answer)


In [ ]:
df.apply(build_vocab,axis=1)

['what', 'is', 'the', 'capital', 'of', 'france'] ['paris']
['what', 'is', 'the', 'capital', 'of', 'germany'] ['berlin']
['who', 'wrote', "'to", 'kill', 'a', "mockingbird'"] ['harper-lee']
['what', 'is', 'the', 'largest', 'planet', 'in', 'our', 'solar', 'system'] ['jupiter']
['what', 'is', 'the', 'boiling', 'point', 'of', 'water', 'in', 'celsius'] ['100']
['who', 'painted', 'the', 'mona', 'lisa'] ['leonardo-da-vinci']
['what', 'is', 'the', 'square', 'root', 'of', '64'] ['8']
['what', 'is', 'the', 'chemical', 'symbol', 'for', 'gold'] ['au']
['which', 'year', 'did', 'world', 'war', 'ii', 'end'] ['1945']
['what', 'is', 'the', 'longest', 'river', 'in', 'the', 'world'] ['nile']
['what', 'is', 'the', 'capital', 'of', 'japan'] ['tokyo']
['who', 'developed', 'the', 'theory', 'of', 'relativity'] ['albert-einstein']
['what', 'is', 'the', 'freezing', 'point', 'of', 'water', 'in', 'fahrenheit'] ['32']
['which', 'planet', 'is', 'known', 'as', 'the', 'red', 'planet'] ['mars']
['who', 'is', 'the', 'au

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [ ]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 "'to": 12,
 'kill': 13,
 'a': 14,
 "mockingbird'": 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 "'1984'": 67,
 'george-orwell': 68,
 'currency': 69,
 '

In [ ]:
len(vocab)

326

In [ ]:
# Conveet into numerical index
def text_to_indices(text, vocab):
  indexed_text=[]
  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])
  return indexed_text

In [ ]:
text_to_indices("What is the capital of prince?",vocab)

[1, 2, 3, 4, 5, 0]

In [ ]:
import torch
from torch.utils.data import Dataset,DataLoader

In [ ]:
class QADataset(Dataset):
  def  __init__(self, df,vocab):
    self.df= df
    self.vocab=vocab
  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):
    numerical_question= text_to_indices(self.df.iloc[index]['question'],self.vocab)
    numerical_answer= text_to_indices(self.df.iloc[index]['answer'],self.vocab)

    return torch.tensor(numerical_question),torch.tensor(numerical_answer)

In [ ]:
dataset= QADataset(df,vocab)

In [ ]:
dataset[9]

(tensor([ 1,  2,  3, 50, 51, 19,  3, 45]), tensor([52]))

In [ ]:
dataloader= DataLoader(dataset,batch_size=1,shuffle=True)

In [ ]:
for question,answer in dataloader:
  print(question)
  print(answer)

tensor([[ 42, 137, 118,   3, 249,   5, 250]])
tensor([[251]])
tensor([[ 42,  18, 118,   3, 187, 188]])
tensor([[189]])
tensor([[10, 75, 76]])
tensor([[77]])
tensor([[  1,   2,   3, 181, 182, 183, 184]])
tensor([[185]])
tensor([[  1,   2,   3, 103,   5, 104,  19, 105]])
tensor([[106]])
tensor([[ 10, 140,   3, 141, 142, 143, 144,  83,   3, 145]])
tensor([[146]])
tensor([[ 42, 201,   2,  14, 202, 203, 204, 205]])
tensor([[206]])
tensor([[ 42, 137,   2, 227, 143,   3, 228, 229]])
tensor([[156]])
tensor([[  1,   2,   3, 213,   5,  14, 214, 215]])
tensor([[216]])
tensor([[  1,   2,   3,   4,   5, 288]])
tensor([[289]])
tensor([[ 42, 168,   2,   3,  17, 169, 170]])
tensor([[171]])
tensor([[ 10, 140,   3, 141, 272,  93, 273,   5,   3, 274]])
tensor([[275]])
tensor([[  1,   2,   3,   4,   5, 281]])
tensor([[282]])
tensor([[ 42, 320,   2,  62,  63,   3, 321,   5, 322]])
tensor([[323]])
tensor([[ 78,  79, 263, 152,  14, 264, 154]])
tensor([[36]])
tensor([[ 1,  2,  3, 92, 93, 94]])
tensor([[95]])


**RNN Architecture**

1 Hiddden layer

embedding layers 50 dimensions nwurons

input :50

hidden node :64

output neurons = total no of unique word= 326



In [ ]:
import torch.nn as nn


In [ ]:
class SimpleRNN(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embedding= nn.Embedding(vocab_size,50)
    self.rnn= nn.RNN(50,64,batch_first=True)
    self.fc= nn.Linear(64,vocab_size)

  def forward(self,question):
     embedded_question= self.embedding(question)
     hidden,final= self.rnn(embedded_question)
     output= self.fc(final.squeeze(0))
     return output


In [ ]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [ ]:
dataset[0][0]

tensor([1, 2, 3, 4, 5, 6])

In [ ]:
x= nn.Embedding(326,embedding_dim=50)

In [ ]:
x(dataset[0][0]).shape

torch.Size([6, 50])

In [ ]:
a=x(dataset[0][0])
a

tensor([[ 1.3878, -0.5097,  0.9714,  0.4275,  0.2133,  0.1987, -0.2066, -1.0599,
          0.9745,  0.6019,  0.6312,  1.9109, -1.0838, -2.3717, -0.3271, -0.9745,
         -0.6795,  0.9482, -0.8393,  0.3333, -0.2326,  1.7608,  0.2431,  0.7637,
         -0.2915,  0.7062,  0.8014, -0.7105,  0.7665, -2.1762, -0.1654,  0.0627,
         -0.5568,  0.3721,  0.2441,  0.8657,  0.5548,  1.2261,  0.3596,  1.1864,
         -0.4869, -1.4914, -0.3378, -0.9150,  1.5067, -0.8239,  0.4934,  0.2048,
         -2.2074,  1.0418],
        [-0.3428,  1.8485,  1.9431,  0.8385, -0.4244,  1.6165, -0.1236, -0.1755,
          0.8229, -0.3226, -0.6104,  0.4015, -0.0379,  2.2622,  0.7406, -2.2606,
         -0.2557, -0.5371, -0.7922,  0.2448, -0.0493, -1.7154, -0.5713,  0.3162,
         -0.1998,  0.5611,  0.5044,  2.0306,  0.2209, -0.1869, -1.0481, -1.1075,
          1.5896,  2.1494, -0.2181, -0.6091,  0.8686, -1.3651, -1.2352,  0.0701,
          1.6205,  0.7181, -0.3668, -1.4535,  1.7570,  0.0705,  0.9510, -1.4603,


In [ ]:
y=nn.RNN(50,64)

In [ ]:
y(a)

(tensor([[ 0.1108,  0.4846,  0.1447, -0.2413, -0.4205, -0.4411,  0.4958,  0.1356,
           0.3548, -0.5753, -0.2816, -0.1660, -0.3626, -0.5337,  0.5549, -0.2131,
          -0.0379, -0.4379,  0.5604,  0.2897,  0.0483, -0.4647,  0.0018,  0.7074,
          -0.6724,  0.5088,  0.0634, -0.7017, -0.1423,  0.3062,  0.6490,  0.7810,
           0.0617,  0.3941, -0.3137,  0.1339, -0.0703, -0.3644,  0.4627,  0.1115,
           0.5387,  0.4555,  0.2190, -0.5839, -0.2610,  0.1268,  0.6321,  0.8428,
           0.1171, -0.3356,  0.3763,  0.1014,  0.2303, -0.3542, -0.5363, -0.5621,
           0.5743, -0.1235, -0.0975, -0.0985, -0.5104,  0.1971,  0.0292,  0.5176],
         [ 0.7699,  0.5358, -0.0132, -0.2759, -0.2838, -0.1431, -0.2067, -0.0237,
           0.8136, -0.8335,  0.8621, -0.5405,  0.5012, -0.4703,  0.2243, -0.0733,
           0.2478,  0.4568, -0.5935,  0.5957,  0.5987,  0.5975,  0.6345, -0.1228,
          -0.0977,  0.6715, -0.8179, -0.9720, -0.4434, -0.3976, -0.0745,  0.4986,
          -0.65

In [ ]:
# Rnn Hidden Stats
y(a)[0]  # hidden output wrt to timestamp

tensor([[ 0.1108,  0.4846,  0.1447, -0.2413, -0.4205, -0.4411,  0.4958,  0.1356,
          0.3548, -0.5753, -0.2816, -0.1660, -0.3626, -0.5337,  0.5549, -0.2131,
         -0.0379, -0.4379,  0.5604,  0.2897,  0.0483, -0.4647,  0.0018,  0.7074,
         -0.6724,  0.5088,  0.0634, -0.7017, -0.1423,  0.3062,  0.6490,  0.7810,
          0.0617,  0.3941, -0.3137,  0.1339, -0.0703, -0.3644,  0.4627,  0.1115,
          0.5387,  0.4555,  0.2190, -0.5839, -0.2610,  0.1268,  0.6321,  0.8428,
          0.1171, -0.3356,  0.3763,  0.1014,  0.2303, -0.3542, -0.5363, -0.5621,
          0.5743, -0.1235, -0.0975, -0.0985, -0.5104,  0.1971,  0.0292,  0.5176],
        [ 0.7699,  0.5358, -0.0132, -0.2759, -0.2838, -0.1431, -0.2067, -0.0237,
          0.8136, -0.8335,  0.8621, -0.5405,  0.5012, -0.4703,  0.2243, -0.0733,
          0.2478,  0.4568, -0.5935,  0.5957,  0.5987,  0.5975,  0.6345, -0.1228,
         -0.0977,  0.6715, -0.8179, -0.9720, -0.4434, -0.3976, -0.0745,  0.4986,
         -0.6536, -0.0414, 

In [ ]:
b= y(a)[1]  # final output
b

tensor([[-0.1529,  0.8244,  0.1060,  0.1901,  0.7888, -0.7220,  0.0244,  0.2847,
         -0.5230,  0.0029,  0.5762,  0.0560,  0.3088, -0.8512, -0.2704, -0.7346,
         -0.7477, -0.8828,  0.7125,  0.4292,  0.0094, -0.3109,  0.4585,  0.5325,
         -0.6674, -0.3158, -0.1142, -0.0696,  0.4280,  0.6806,  0.5314, -0.0975,
          0.7621,  0.7035, -0.7138, -0.0946,  0.4745,  0.3055,  0.1380, -0.3969,
          0.5363, -0.8238,  0.6377, -0.3301,  0.6226, -0.4041,  0.8127,  0.2988,
          0.4633,  0.1535, -0.5371,  0.4771,  0.6462, -0.5255,  0.3347, -0.9260,
          0.4416, -0.1366, -0.0311, -0.5141, -0.2013,  0.7225,  0.5991,  0.7533]],
       grad_fn=<SqueezeBackward1>)

In [ ]:
z= nn.Linear(64,326)
z

Linear(in_features=64, out_features=326, bias=True)

In [ ]:
z(b)

tensor([[ 1.1775e-01, -2.9113e-01, -2.2815e-01,  1.6919e-01, -1.0926e-01,
         -1.9259e-01, -1.1151e-01,  2.1053e-01,  1.6097e-01, -4.0086e-01,
         -2.3235e-02,  3.4714e-01, -1.9931e-01, -6.4126e-02,  3.0785e-01,
         -3.3484e-01,  2.3484e-01,  2.1193e-01, -8.4577e-02, -4.1982e-02,
         -1.1275e-02, -2.8693e-03, -1.1274e-01,  6.1854e-02, -4.2460e-01,
          6.3924e-04, -3.3299e-01,  2.5712e-01, -1.1244e-01,  1.6047e-02,
         -4.3801e-01,  3.4009e-02,  4.1259e-01, -1.5946e-01, -6.1794e-01,
          9.9170e-02, -3.1646e-01, -3.2908e-02, -6.3333e-01,  4.0247e-02,
          1.3192e-01, -2.7440e-01,  3.8574e-01, -2.9847e-02, -8.7932e-03,
          3.9193e-01, -2.6665e-01, -1.5180e-01, -9.6261e-02,  3.1084e-02,
          2.8964e-01,  6.9228e-02,  8.4083e-02, -5.4726e-01, -3.0873e-01,
          4.5294e-02,  1.7652e-02, -3.7573e-02, -2.3478e-01, -6.6994e-02,
         -7.7291e-02, -2.6233e-02,  8.2624e-02,  3.4272e-01,  5.4276e-01,
          2.7205e-01, -2.6149e-01, -1.

In [ ]:
z(b).shape

torch.Size([1, 326])

In [ ]:
learning_rate= 0.001
epochs=20

In [ ]:
model= SimpleRNN(len(vocab))

In [ ]:
criterion= nn.CrossEntropyLoss()
optimizer= torch.optim.Adam(model.parameters(),lr=learning_rate)

In [ ]:
# Training loop
for epoch in range(epochs):
  total_loss=0
  for question,answer in dataloader:
    optimizer.zero_grad()

    # forward pass
    output= model(question)
    # calculate loss
    loss= criterion(output,answer[0])
    # gradients
    loss.backward()
    #update
    optimizer.step()
    # calculate total loss
    total_loss= total_loss + loss.item()

  print(f"Epoch: {epoch +1}, Loss:{total_loss:4f}")

Epoch: 1, Loss:518.866976
Epoch: 2, Loss:450.721770
Epoch: 3, Loss:372.662608
Epoch: 4, Loss:310.417091
Epoch: 5, Loss:256.967411
Epoch: 6, Loss:208.579506
Epoch: 7, Loss:164.292683
Epoch: 8, Loss:126.752439
Epoch: 9, Loss:97.035410
Epoch: 10, Loss:73.589808
Epoch: 11, Loss:56.917022
Epoch: 12, Loss:43.817918
Epoch: 13, Loss:34.805841
Epoch: 14, Loss:28.051549
Epoch: 15, Loss:22.690211
Epoch: 16, Loss:19.023220
Epoch: 17, Loss:15.888299
Epoch: 18, Loss:13.485676
Epoch: 19, Loss:11.638974
Epoch: 20, Loss:10.116184


In [ ]:
# Prediction
def predict(model,question, threshold=0.5):
  # convert text into convet
 numerical_question= text_to_indices(question,vocab)
# print(numerical_question)

 # convert numerical to tensor
 question_tensor= torch.tensor(numerical_question).unsqueeze(0)
# print(question_tensor)

 # forward pass to model
 output= model(question_tensor)
 #print(output)

 # convert logits to probability
 probs= torch.nn.functional.softmax(output, dim=1)
 # find index of max probs
 value, index= torch.max(probs, dim=1)
 if value < threshold:
   print("I dont know")
 else:
  print(list(vocab.keys())[index])
 #print(value, index)


In [ ]:
predict(model,"what is prince")

I dont know


In [ ]:
list(vocab.keys())[16]

'harper-lee'

In [ ]:
predict(model,"what is the capital of France")

paris


In [ ]:
list(vocab.keys())[7]

'paris'

In [ ]:
predict(model,"what is the capital of France")

paris
